# Import libraries

In [46]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
import re
import pandas as pd

# Cleaning functions

In [48]:
# Function to remove the first row (duplicated original column names)
def remove_first_row(df):
    return df.iloc[1:, :].copy()

# Usage:
# subset_1_information_samples = remove_first_row(subset_1_information_samples)





# Function to clean and standardize lab/sample IDs to FXX-XXXX format
def clean_lab_ids(series):
    return (
        series
        .astype(str)
        .str.replace(r'\s+', '-', regex=True)                 # Replace spaces with -
        .str.replace(r'^(F\d{2})(\d+)', r'\1-\2', regex=True) # Format FXX-XXXX
    )


# Usage:
# subset_1_information_samples['10_ciat_lab_id'] = clean_lab_ids(subset_1_information_samples['10_ciat_lab_id'])



# Function to clean and standardize Gene Bank - Breeding program IDs
def clean_standardize_ids(series):
    return (
        series
        .astype(str)
        .str.replace(r'-1$', '', regex=True)                   # Remove trailing -1
        .str.strip()                                           # Remove leading/trailing spaces
        .str.replace(r'[_\s]+', '-', regex=True)               # Replace spaces/underscores with -
        .str.replace(r'([A-Za-z])(\d)', r'\1-\2', regex=True)  # Letter followed by number
        .str.replace(r'(\d)([A-Za-z])', r'\1-\2', regex=True)  # Number followed by letter
        .str.replace(r'-+', '-', regex=True)                   # Remove repeated -
        .str.replace(r'^-|-$', '', regex=True)                 # Remove leading/trailing -
        .str.replace('ABC-', 'CIAT-', regex=False)             # Replace id's strings 'ABC' with 'CIAT'
    )

    # subset_1_information_samples['10_gene_bank_breeding_program_id'] = clean_standardize_ids(subset_1_information_samples['10_gene_bank_breeding_program_id'])


# 1. Shiny app training set
Requested by: Khaled Al-Sham'aa, (ICARDA)

Date: 2026_05_19

In [ ]:
subsets_gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_sorted_by_sql.csv')
subsets_gas.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,1,3,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,62.577353,4.000526,11.156690,14.2,17.82863877,135.945555,227.8476704,37.051645,65.414752
1,1,4,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,61.249843,4.225907,11.205181,15,18.29422035,133.168047,270.0095702,38.185077,63.799940
2,1,5,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,29.500225,63.019856,4.071031,10.976075,13.8,17.41685196,137.016371,249.4607864,37.088353,64.343484
3,1,6,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,30.827735,67.002386,4.285055,11.302938,13.9,16.86945515,148.741403,175.9559634,41.714068,60.152044
4,1,7,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,29.500225,66.117380,4.130032,11.233760,14,16.99063023,146.659412,149.5197103,39.832832,62.557335


In [ ]:
# Filter by functional group 'Grass'
subsets_gas_grass = subsets_gas[subsets_gas['functional_group'] == 'Grass']
subsets_gas_grass.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
39,1,44,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,26.019199,58.536668,2.367747,6.302361,9.1,10.76651796,122.759406,194.6816788,27.337315,48.347519
40,1,45,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,27.789212,61.806682,2.445451,6.425495,8.8,10.39611646,129.591169,117.6219867,25.413695,53.012554
41,1,46,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,25.576695,62.094165,2.506516,6.742543,9.8,10.85857685,130.141925,195.7779286,28.971955,48.776690
42,1,47,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,17.169131,44.186601,1.837097,7.078486,10.7,16.01953085,92.531478,89.5309162,25.377242,58.411031
43,1,48,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,20.709158,51.726628,2.360844,8.502303,11.4,16.43699461,108.342750,110.6977836,30.734726,57.941924


In [ ]:
# Filter by subset '2'
subset_2_gas_grass = subsets_gas_grass[subsets_gas_grass['subset'] == 2]
subset_2_gas_grass.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
2373,2,101,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,20.945160,58.779199,3.246500,10.094461,15.5,18.1,NaN,NaN,NaN,NaN
2374,2,102,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,20.060153,55.239172,3.089264,9.667740,15.4,18.7,117.726263,#DIV/0!,164.369645,12.535150
2375,2,103,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,20.060153,57.451689,3.169504,10.161721,15.8,18.7,122.564018,#DIV/0!,60.146889,36.042450
2472,2,206,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,19.396398,47.915741,2.967649,7.901495,15.3,17.3,102.159310,#DIV/0!,47.734635,35.291931
2473,2,207,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,19.838901,50.039757,2.995674,8.099619,15.1,16.9,106.687844,#DIV/0!,52.077666,33.159869


In [ ]:
subset_2_gas_grass.batch.unique()

array([34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61])

In [ ]:
# Filter by batch =< 40
subset_2_gas_grass_batchs = subset_2_gas_grass[subset_2_gas_grass['batch'] >= 50]
subset_2_gas_grass_batchs.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
3870,2,1701,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,36.432778,79.886610,4.881992,12.225690,13.4,16.9,155.827612,34.19146976,43.344262,55.018936
3871,2,1702,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,36.875281,82.099127,5.125664,12.723270,13.9,16.8,155.999232,35.00485836,43.501476,55.574898
3872,2,1703,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,35.105268,79.444106,5.160474,12.653738,14.7,16.9,153.594861,35.25395301,43.886070,55.745158
3873,2,1704,Breding,F25-2078,CIAT-PM-21-6106,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,35.105268,75.019073,5.090264,12.194921,14.5,17.8,148.711930,42.1835579,39.267646,61.562784
3874,2,1705,Breding,F25-2078,CIAT-PM-21-6106,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,39.530302,82.984133,5.652833,13.170346,14.3,17.3,160.584546,36.28321101,44.216137,57.640147


In [ ]:
#subset_2_gas_grass_batchs.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/ciat_gas_for_shiny_app_and_blues.csv', index = None)

# 2. Gas dashboard of Subset 4
Requested by: Alejandra Marín

Date: 2026_05_22

In [ ]:
dashboard_small = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_dashboard_small.csv')
dashboard_small.head()

,subset,id_lab,id,tax_name,functional_group,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,1,F24-3469,ABC-10647,Stylosanthes scabra,Herbaceous_legumes,15.90,18.79,48.31,67.34
1,1,F24-3470,ABC-11194,Stylosanthes hamata,Herbaceous_legumes,15.17,18.02,46.12,59.25
2,1,F24-3471,ABC-11999,Stylosanthes guianensis,Herbaceous_legumes,12.98,16.25,46.61,58.55
3,1,F24-3472,ABC-12318,Stylosanthes hamata,Herbaceous_legumes,13.99,16.96,52.27,56.88
4,1,F24-3427,ABC-1257,Stylosanthes scabra,Herbaceous_legumes,14.53,16.40,43.29,58.67


In [ ]:
# Filter data of subset 4
dashboard_small_subset_4 = dashboard_small[dashboard_small['subset'] == 4]

In [ ]:
# Order by id
def natural_sort_key(s):
    """Splits a string into alphanumeric components and converts numbers to integers for natural sorting."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', str(s))]

# Apply natural sort to the 'id' column
dashboard_small_subset_4_natural_sorted = dashboard_small_subset_4.sort_values(
    by='id',
    key=lambda col: col.apply(natural_sort_key)
).reset_index(drop=True)

display(dashboard_small_subset_4_natural_sorted.head())

,subset,id_lab,id,tax_name,functional_group,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,4,F25-2640,Cayman-Br-02-1752,Urochloa interespecifico,Grass,12.99,13.71,50.17,50.22
1,4,F25-2574,CIAT-326,Desmodium scorpiurus,Herbaceous_legumes,13.86,16.66,55.45,39.09
2,4,F25-2575,CIAT-415,Vigna radiata,Herbaceous_legumes,11.22,15.38,43.25,51.39
3,4,F25-2576,CIAT-416,Vigna radiata,Herbaceous_legumes,12.38,16.84,50.26,56.45
4,4,F25-2577,CIAT-517,Macroptilium atropurpureum,Herbaceous_legumes,11.58,14.88,52.88,37.99


In [ ]:
#dashboard_small_subset_4_natural_sorted.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/subset_4_gas_dashboard_small.csv', index = None)

# 3. Grasses gas complete

Requested by: Claudia Perea
Date: 2026_05_26

In [ ]:
subsets_gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_sorted_by_sql.csv')
subsets_gas.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,1,3,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,62.577353,4.000526,11.156690,14.2,17.82863877,135.945555,227.8476704,37.051645,65.414752
1,1,4,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,61.249843,4.225907,11.205181,15,18.29422035,133.168047,270.0095702,38.185077,63.799940
2,1,5,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,29.500225,63.019856,4.071031,10.976075,13.8,17.41685196,137.016371,249.4607864,37.088353,64.343484
3,1,6,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,30.827735,67.002386,4.285055,11.302938,13.9,16.86945515,148.741403,175.9559634,41.714068,60.152044
4,1,7,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,29.500225,66.117380,4.130032,11.233760,14,16.99063023,146.659412,149.5197103,39.832832,62.557335


In [ ]:
# Filter by functional group 'Grass'
grasses = subsets_gas[subsets_gas['functional_group'] == 'Grass']
grasses.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
39,1,44,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,26.019199,58.536668,2.367747,6.302361,9.1,10.76651796,122.759406,194.6816788,27.337315,48.347519
40,1,45,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,27.789212,61.806682,2.445451,6.425495,8.8,10.39611646,129.591169,117.6219867,25.413695,53.012554
41,1,46,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,25.576695,62.094165,2.506516,6.742543,9.8,10.85857685,130.141925,195.7779286,28.971955,48.776690
42,1,47,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,17.169131,44.186601,1.837097,7.078486,10.7,16.01953085,92.531478,89.5309162,25.377242,58.411031
43,1,48,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,20.709158,51.726628,2.360844,8.502303,11.4,16.43699461,108.342750,110.6977836,30.734726,57.941924


In [ ]:
#grasses.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/grasses.csv', index = None)

# 4. Primary traits for metabolomics

Requested by : Jenny Gallo

Date: 2026_06_03


In [ ]:
requested = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/requested_jenny.csv')
data = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_dashboard_complete.csv')

In [ ]:
requested = remove_first_row(requested)
requested.head(50)

,approach,id,tax_name,functional_group,ch4_intensity_ml_g_tddm,tddm_percentage,ch4_intensity_decrease,ch4_percentage_8h,ch4_percentage_24h,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
2,Approach 1 (a),CIAT-19213,Clitoria ternatea,Climber,39.1,63.7,39%,14.98888889,16.08929282,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Approach 2,CIAT-9434,Clitoria ternatea,Climber,46.51,51.78,33%,14.36666667,15.85424307,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Approach 2,CIAT-955,Clitoria ternatea,Climber,45.45,59.65,30%,12.82222222,14.54174742,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Approach 4,CIAT-9432,Clitoria ternatea,Climber,57.24,50.87,11%,17.36,18.90480266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Approach 4,CIAT-18447,Clitoria ternatea,Climber,53.0,55.3,18%,14.51111111,16.92415425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Approach 1 (a),CIAT-7317,Canavalia sp.,Climber,33.0,69.5,49%,14.86111111,16.36328667,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Approach 1 (a),CIAT-8719,Canavalia sp.,Climber,35.8,69.0,45%,14.66388889,16.92290927,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Approach 2,CIAT-20803,Canavalia sp.,Climber,42.61,55.70,34%,13.8,15.97697193,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,Approach 4,CIAT-19032,Canavalia sp.,Climber,65.33,43.83,6%,13.875,16.02639497,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,Approach 1 (a),CIAT-15154,Centrosema molle,Climber,38.17,61.95,41%,14,16.05586855,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
data.head()

,subset,requisitioner,no,id_lab,id,n_replicates,tax_order,family,genus,species,...,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_8h_percentage,ch4_24h_percentage,part_fact,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,1,LMF,2202,F25-0769,AMC-ICARDA-156302,3,Fabales,Fabaceae,Trifolium,repens,...,86.99,6.03,14.08,15.97,16.37,3.75,30.99,-75.10,43.67,72.00
1,1,LMF,2235,F25-0770,AMC-ICARDA-165072,3,NaN,NaN,Vicia,tenuifolia,...,86.99,6.19,14.43,14.37,18.77,2.97,31.76,NaN,56.12,56.78
2,1,LMF,2226,F25-0773,AMC-ICARDA-168125,3,NaN,NaN,Lathyrus,sylvestris,...,61.77,4.70,10.30,15.53,17.77,3.12,22.66,NaN,55.10,42.32
3,1,Genetic bank,1057,F24-3469,CIAT-10647,6,Fabales,Fabaceae,Stylosanthes,scabra,...,79.20,6.90,14.88,15.90,22.28,3.95,32.01,151.04,48.31,67.34
4,1,Genetic bank,1060,F24-3470,CIAT-11194,6,Fabales,Fabaceae,Stylosanthes,hamata,...,70.41,5.14,12.69,15.17,20.65,3.91,27.31,184.66,46.12,59.25


In [ ]:
requested.rename(columns={'gene_bank_breeding_id': 'id'}, inplace=True)
requested['id'] = clean_standardize_ids(requested['id'])
approach_id  = requested.iloc[:, :2]
approach_id.head(50)

,approach,id
2,Approach 1 (a),CIAT-19213
3,Approach 2,CIAT-9434
4,Approach 2,CIAT-955
5,Approach 4,CIAT-9432
6,Approach 4,CIAT-18447
7,Approach 1 (a),CIAT-7317
8,Approach 1 (a),CIAT-8719
9,Approach 2,CIAT-20803
10,Approach 4,CIAT-19032
11,Approach 1 (a),CIAT-15154


In [ ]:
merged_df = approach_id.merge(data, on='id')
merged_df.head(50)

,approach,id,subset,requisitioner,no,id_lab,n_replicates,tax_order,family,genus,...,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_8h_percentage,ch4_24h_percentage,part_fact,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,Approach 1 (a),CIAT-19213,1,Genetic bank,1649,F24-3501,9,Fabales,Fabaceae,Clitoria,...,71.89,7.58,11.57,14.99,18.66,4.15,24.65,194.95,39.14,63.66
1,Approach 2,CIAT-9434,2,Genetic bank,83,F24-3635,9,Fabales,Fabaceae,Clitoria,...,81.72,6.21,13.39,14.89,17.69,3.03,29.55,3.51,56.61,54.00
2,Approach 2,CIAT-955,1,Genetic bank,32,F24-3425,9,Fabales,Fabaceae,Clitoria,...,84.29,4.83,12.25,12.82,15.90,3.29,26.67,138.42,45.45,59.65
3,Approach 4,CIAT-9432,1,Genetic bank,1039,F24-3463,6,Fabales,Fabaceae,Clitoria,...,70.80,6.57,13.41,17.47,20.60,3.57,29.38,-897.64,54.34,55.33
4,Approach 4,CIAT-9432,3,Breeding,1002,F25-2635,18,Fabales,Fabaceae,Clitoria,...,84.52,7.57,13.71,15.25,17.61,2.72,28.73,100.01,60.36,48.13
5,Approach 4,CIAT-18447,1,Genetic bank,1218,F24-3484,9,Fabales,Fabaceae,Clitoria,...,80.39,6.57,13.64,14.51,20.06,3.20,29.34,361.74,53.02,55.28
6,Approach 1 (a),CIAT-7317,1,Genetic bank,185,F24-3435,36,Fabales,Fabaceae,Canavalia,...,64.66,5.20,10.57,14.86,18.14,5.01,22.85,108.22,32.97,69.48
7,Approach 1 (a),CIAT-8719,1,Genetic bank,347,F24-3453,30,Fabales,Fabaceae,Canavalia,...,65.97,5.12,10.94,14.29,18.98,4.97,23.39,239.49,34.19,69.74
8,Approach 2,CIAT-20803,1,Genetic bank,1905,F24-3512,9,Fabales,Fabaceae,Canavalia,...,68.27,4.88,10.92,13.80,18.37,3.86,23.37,344.33,42.61,55.70
9,Approach 4,CIAT-19032,2,Genetic bank,339,F24-3647,9,Fabales,Fabaceae,Canavalia,...,80.83,5.02,12.39,13.27,17.19,3.39,26.81,-2.50,45.70,59.15


In [ ]:
merged_df.columns

Index(['approach', 'id', 'subset', 'requisitioner', 'no', 'id_lab',
       'n_replicates', 'tax_order', 'family', 'genus', 'species', 'tax_name',
       'functional_group', 'set_ciat', 'batch', 'run', 'replication',
       'syrange', 'sample_weight_g', 'undigested_dm_g', 'dm_incubated',
       'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml', 'ch4_8h_ml',
       'ch4_24h_ml', 'ch4_8h_percentage', 'ch4_24h_percentage', 'part_fact',
       'ch4_ml_g_dm_incubaed_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm'],
      dtype='object')

In [ ]:
CIAT-Mulato_II	Brachiaria interespecifico
CIAT-BR02_1752	Brachiaria interespecifico
CIAT-BR15_2891	Brachiaria interespecifico
CIAT-BR19_5029	Brachiaria interespecifico


In [ ]:
primary_traits = merged_df[['approach', 'id','tax_name','functional_group', 'dm_incubated',  'digested_feed_mg', 'ch4_24h_ml']]
primary_traits.head(50)

,approach,id,tax_name,functional_group,dm_incubated,digested_feed_mg,ch4_24h_ml
0,Approach 1 (a),CIAT-19213,Clitoria ternatea,Herbaceous_legumes,469.33,298.75,11.57
1,Approach 2,CIAT-9434,Clitoria ternatea,Herbaceous_legumes,454.10,236.60,13.39
2,Approach 2,CIAT-955,Clitoria ternatea,Herbaceous_legumes,459.34,274.00,12.25
3,Approach 4,CIAT-9432,Clitoria ternatea,Herbaceous_legumes,456.56,252.63,13.41
4,Approach 4,CIAT-9432,Clitoria ternatea,Herbaceous_legumes,477.33,229.65,13.71
5,Approach 4,CIAT-18447,Clitoria ternatea,Herbaceous_legumes,464.97,257.04,13.64
6,Approach 1 (a),CIAT-7317,Canavalia sp.,Herbaceous_legumes,462.50,321.35,10.57
7,Approach 1 (a),CIAT-8719,Canavalia sp.,Herbaceous_legumes,467.83,326.25,10.94
8,Approach 2,CIAT-20803,Canavalia sp.,Herbaceous_legumes,467.17,260.21,10.92
9,Approach 4,CIAT-19032,Canavalia sp.,Herbaceous_legumes,462.32,273.47,12.39


In [ ]:
#primary_traits.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/primary_traits_to_iomicas.csv', index=None)

# 5. Grasses for Breeding
second database

Requested by: Claudia Perea

Date: 2026_05_26

## Gas

### Data load

In [14]:
df = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/05_grasses_for_breeding/gas_clean_subsets_1234_2026_06_09.csv')
df.head(2)


,id_lab,batch,subset,no,requisitioner,id,tax_order,family,genus,species,...,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2,delete
0,F24-3416,1,1,3,Genetic bank,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,...,227.8476704,37.051645,65.414752,NaN,"1,2,3",NaN,NaN,NaN,NaN,no
1,F24-3416,1,1,4,Genetic bank,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,...,270.0095702,38.185077,63.799940,NaN,"1,2,3",NaN,NaN,NaN,NaN,no


### Filtering and formating

In [15]:
df.requisitioner.unique()

array(['Genetic bank', 'LMF', 'Breeding', 'LMF-invivo', 'Benchmark',
       'Isabel Molina', 'Mauricio Sotelo',
       'Jacobo Arango/Alejandro Montoya',
       'Jacobo Arango/ Alejandro Montoya'], dtype=object)

In [16]:
df2 = df[df['requisitioner'] == 'Breeding']

In [17]:
df2.columns

Index(['id_lab', 'batch', 'subset', 'no', 'requisitioner', 'id', 'tax_order',
       'family', 'genus', 'species', 'tax_name', 'functional_group',
       'set_ciat', 'run', 'replication', 'syrange', 'sample_weight_g',
       'undigested_dm_g', 'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml',
       'net_gas_24h_ml', 'ch4_8h_ml', 'ch4_24h_ml', 'ch4_8h_percentage',
       'ch4_24h_percentage', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubaed_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete'],
      dtype='object')

In [18]:
df3 = df2[['subset','no','requisitioner','id_lab','id','tax_name','functional_group','batch','run','replication','syrange','ch4_8h_ml','ch4_24h_ml','tddm']]
df3 = df3.copy()

cols = ['ch4_8h_ml', 'ch4_24h_ml', 'tddm']
df3[cols] = df3[cols].round(2)

In [19]:
df3.head()

,subset,no,requisitioner,id_lab,id,tax_name,functional_group,batch,run,replication,syrange,ch4_8h_ml,ch4_24h_ml,tddm
423,1,462,Breeding,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,10,1,1,54.0,4.98,10.56,66.86
424,1,463,Breeding,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,10,1,2,55.0,5.02,10.10,65.35
425,1,464,Breeding,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,10,1,3,56.0,5.05,10.36,65.88
426,1,465,Breeding,F24-3583,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,10,1,1,57.0,5.07,10.65,63.04
427,1,466,Breeding,F24-3583,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,10,1,2,58.0,5.19,10.43,65.25


In [20]:
df3.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/gas_breeding_complete.csv', index=None)

### Average values

In [21]:
#Columns processing

category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_name', 'functional_group', 'batch',
       'run', 'replication', 'syrange']
numeric_columns = ['ch4_8h_ml', 'ch4_24h_ml', 'tddm']

for df in [df3]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [22]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_gas'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [12]:
df4 = mean_with_replicates(df3, category_columns, numeric_columns)
df4.head(2)

/tmp/ipykernel_63224/1161671181.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_63224/1161671181.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_63224/1161671181.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


,id_lab,subset,no,requisitioner,id,tax_name,functional_group,batch,run,replication,syrange,ch4_8h_ml,ch4_24h_ml,tddm,n_replicates_gas
0,F24-3582,1,462,Breeding,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,10,1,1,54.0,5.17,10.95,67.39,9
1,F24-3583,1,465,Breeding,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,10,1,1,57.0,5.32,11.11,63.97,9


In [23]:
df4.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/gas_breeding_average.csv', index=None)

## Nutrition

### Data load

In [42]:
nu = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/05_grasses_for_breeding/nutrition_complete_1234_2026_06_09.csv')
nu.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,1,1.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,91.930807,12.725691,87.274309,33.010522,28.489614,45.210329
1,1,2.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,92.010000,12.629062,87.370938,33.010522,27.612657,45.235119


### Filtering and formating

In [43]:
nu2 = nu[nu['requisitioner'] == 'Breeding']
nu2.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
52,1,51.0,Breeding,F24-3582,CIAT-BR-02-1752,Poales,Poaceae,Urochloa,interespecifico,Brachiaria interespecifico,Grass,4,95.980000,15.641696,84.358304,10.917973,21.195781,55.590042
53,1,52.0,Breeding,F24-3582,CIAT-BR-02-1752,Poales,Poaceae,Urochloa,interespecifico,Brachiaria interespecifico,Grass,4,96.070786,15.763510,84.236490,10.917973,21.085848,55.233675


In [44]:
cols = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']
nu2[cols] = nu2[cols].round(2)

/tmp/ipykernel_63224/1946393332.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nu2[cols] = nu2[cols].round(2)


In [45]:
nu2.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/nutrition_breeding_complete.csv', index=None)

In [27]:
nu2.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm'],
      dtype='object')

In [28]:
nu3 = nu2[['subset','no','requisitioner','id_lab','id','tax_name','functional_group','dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']]
nu3 = nu3.copy()

cols = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']
nu3[cols] = nu3[cols].round(2)

In [29]:
nu3

,subset,no,requisitioner,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
52,1,51.0,Breeding,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,95.98,15.64,84.36,10.92,21.20,55.59
53,1,52.0,Breeding,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,96.07,15.76,84.24,10.92,21.09,55.23
54,1,53.0,Breeding,F24-3583,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,96.97,15.14,84.86,9.39,21.27,54.83
55,1,54.0,Breeding,F24-3583,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,97.12,14.90,85.10,9.39,22.40,56.70
56,1,55.0,Breeding,F24-3584,CIAT-BR-06-0423,Brachiaria interespecifico,Grass,96.56,15.97,84.03,11.01,22.15,57.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1543,3,234.0,Breeding,F25-2639,NaN,Gliricidia sepium,Shrub_Trees,88.99,10.80,89.20,22.52,20.03,42.17
1908,4,NaN,Breeding,F25-2104,CIAT-BR-06-1348,Brachiaria interespecifico,Grass,93.02,14.27,85.73,12.27,25.43,59.24
1909,4,NaN,Breeding,F25-2104,CIAT-BR-06-1348,Brachiaria interespecifico,Grass,93.16,14.68,85.32,12.18,25.85,60.28
1910,4,NaN,Breeding,F25-2166,CIAT-BH-22-0528,Urochloa humidicola,Grass,98.20,12.48,87.52,8.80,36.63,73.37


### Average values

In [30]:
#Columns processing

category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_name', 'functional_group']
numeric_columns = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']

for df in [nu3]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [31]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_nutrition'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [32]:
nu4 = mean_with_replicates(nu3, category_columns, numeric_columns)
nu4.head(2)

/tmp/ipykernel_63224/1194620962.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_63224/1194620962.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_63224/1194620962.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


,id_lab,subset,no,requisitioner,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
0,F24-3582,1,51.0,Breeding,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,96.02,15.70,84.30,10.92,21.14,55.41,2
1,F24-3583,1,53.0,Breeding,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,97.04,15.02,84.98,9.39,21.84,55.76,2


In [33]:
nu4.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/nutrition_breeding_average.csv', index=None)

## Compilation

In [34]:
df4.columns

Index(['id_lab', 'subset', 'no', 'requisitioner', 'id', 'tax_name',
       'functional_group', 'batch', 'run', 'replication', 'syrange',
       'ch4_8h_ml', 'ch4_24h_ml', 'tddm', 'n_replicates_gas'],
      dtype='object')

In [35]:
df5 = df4[['id_lab', 'ch4_8h_ml', 'ch4_24h_ml', 'tddm', 'n_replicates_gas']]

In [36]:
compiled = nu4.merge(df5, on='id_lab', how='left')

In [37]:
# Save the compiled gas and nutrition breeding average data
compiled.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/compilated_gas_nutrition_breeding_average.csv', index=None)

# 6. Stylosanthes Gene Bank

Requested by: Juan José González

Date: 2026_05_25

## Data Load

In [123]:
requested = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/06_stylosanthes_gene_bank/data_requested_juan_jose.csv')
gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/06_stylosanthes_gene_bank/gas_clean_subsets_1234_2026_06_09.csv')
nu = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/06_stylosanthes_gene_bank/nutrition_complete_1234_2026_06_09.csv')

## Filtering and formating

In [124]:
requested['id'] = clean_standardize_ids(requested['id'])

In [125]:
category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'set_ciat','functional_group','batch','run','replication','syrange']
numeric_columns = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm','ch4_8h_ml', 'ch4_24h_ml', 'tddm']

for df in [gas, nu]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [126]:
gas2 = gas[['subset','no','requisitioner','id_lab','id','tax_name','functional_group','batch','run','replication','syrange','ch4_8h_ml','ch4_24h_ml','tddm']]
gas2.round(2)
cols = ['ch4_8h_ml', 'ch4_24h_ml', 'tddm']
gas2[cols] = gas2[cols].round(2)
gas2.head(2)

/tmp/ipykernel_63224/53392924.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gas2[cols] = gas2[cols].round(2)


,subset,no,requisitioner,id_lab,id,tax_name,functional_group,batch,run,replication,syrange,ch4_8h_ml,ch4_24h_ml,tddm
0,1,3,Genetic bank,F24-3416,CIAT-705,Indigofera suffruticosa,Herbaceous_legumes,1,1,1,3.0,4.00,11.16,65.41
1,1,4,Genetic bank,F24-3416,CIAT-705,Indigofera suffruticosa,Herbaceous_legumes,1,1,2,4.0,4.23,11.21,63.80


In [129]:
nu2 = nu[['subset', 'no', 'requisitioner', 'id_lab', 'id', 'functional_group',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']]
nu2.round(2)
cols = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']
nu2[cols] = nu2[cols].round(2)
nu2.head(2)

/tmp/ipykernel_63224/3026070774.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nu2[cols] = nu2[cols].round(2)


,subset,no,requisitioner,id_lab,id,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,1,1.0,Genetic bank,F24-3416,CIAT-705,Herbaceous_legumes,91.93,12.73,87.27,33.01,28.49,45.21
1,1,2.0,Genetic bank,F24-3416,CIAT-705,Herbaceous_legumes,92.01,12.63,87.37,33.01,27.61,45.24


## Compilation

In [127]:
requested_gas = requested.merge(gas2, on='id', how='left')

In [130]:
requested_nutrition = requested.merge(nu2, on='id', how='left')

In [132]:
requested_gas

,gender,species,id,subset,no,requisitioner,id_lab,tax_name,functional_group,batch,run,replication,syrange,ch4_8h_ml,ch4_24h_ml,tddm
0,Stylosanthes,guianensis,CIAT-11313,3,707,Genetic bank,F24-3559,Stylosanthes guianensis,Herbaceous_legumes,52,1,1,176.0,4.20,10.29,48.33
1,Stylosanthes,guianensis,CIAT-11313,3,708,Genetic bank,F24-3559,Stylosanthes guianensis,Herbaceous_legumes,52,1,2,177.0,4.09,10.30,49.15
2,Stylosanthes,guianensis,CIAT-11313,3,709,Genetic bank,F24-3559,Stylosanthes guianensis,Herbaceous_legumes,52,1,3,178.0,3.88,9.65,48.44
3,Stylosanthes,guianensis,CIAT-11313,3,812,Genetic bank,F24-3559,Stylosanthes guianensis,Herbaceous_legumes,53,2,1,65.0,3.60,10.23,32.84
4,Stylosanthes,guianensis,CIAT-11313,3,813,Genetic bank,F24-3559,Stylosanthes guianensis,Herbaceous_legumes,53,2,2,66.0,3.75,10.24,47.18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,Stylosanthes,scabra,CIAT-12710,3,1170,Genetic bank,F24-3580,Stylosanthes scabra,Herbaceous_legumes,55,2,2,99.0,4.66,10.66,51.47
356,Stylosanthes,scabra,CIAT-12710,3,1171,Genetic bank,F24-3580,Stylosanthes scabra,Herbaceous_legumes,55,2,3,100.0,4.73,10.57,46.28
357,Stylosanthes,scabra,CIAT-12710,3,1277,Genetic bank,F24-3580,Stylosanthes scabra,Herbaceous_legumes,56,3,1,98.0,4.82,10.31,49.07
358,Stylosanthes,scabra,CIAT-12710,3,1278,Genetic bank,F24-3580,Stylosanthes scabra,Herbaceous_legumes,56,3,2,99.0,4.74,10.75,42.82


In [131]:
requested_nutrition

,gender,species,id,subset,no,requisitioner,id_lab,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,Stylosanthes,guianensis,CIAT-11313,3,175.0,Genetic bank,F24-3559,Herbaceous_legumes,92.11,11.55,88.45,14.94,38.49,55.73
1,Stylosanthes,guianensis,CIAT-11313,3,176.0,Genetic bank,F24-3559,Herbaceous_legumes,92.26,11.52,88.48,14.94,39.16,56.20
2,Stylosanthes,guianensis,CIAT-11417,3,177.0,Genetic bank,F24-3560,Herbaceous_legumes,91.48,11.93,88.07,12.26,41.32,57.71
3,Stylosanthes,guianensis,CIAT-11417,3,178.0,Genetic bank,F24-3560,Herbaceous_legumes,91.52,12.19,87.81,12.26,41.63,56.74
4,Stylosanthes,guianensis,CIAT-12311,3,247.0,Genetic bank,F24-3572,Herbaceous_legumes,91.67,15.02,84.98,16.63,35.04,52.68
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Stylosanthes,scabra,CIAT-11820,3,190.0,Genetic bank,F24-3566,Herbaceous_legumes,91.89,12.99,87.01,18.06,31.14,52.56
86,Stylosanthes,scabra,CIAT-12485,3,257.0,Genetic bank,F24-3577,Herbaceous_legumes,94.06,13.11,86.89,14.59,30.83,56.54
87,Stylosanthes,scabra,CIAT-12485,3,258.0,Genetic bank,F24-3577,Herbaceous_legumes,93.84,13.58,86.42,14.59,30.72,55.03
88,Stylosanthes,scabra,CIAT-12710,3,263.0,Genetic bank,F24-3580,Herbaceous_legumes,91.68,10.75,89.25,11.50,38.96,58.26


In [134]:
requested_gas.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/06_stylosanthes_gene_bank/stylosanthes_gas_genebank.csv', index =None)

In [135]:
requested_nutrition.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/06_stylosanthes_gene_bank/stylosanthes_nutrition_genebank.csv', index = None)